# Bài toán:

Xây dựng knowledge graph và đánh giá chất lượng câu trả lời dựa của GraphRAG và NaiveRAG trên tập dữ liệu multi-hop QA

**Mục Tiêu:**

- Cài đặt và thử nghiệm GraphRAG và NaiveRAG trên tập dữ liệu multi-hop QA sử dụng thư viện lightrag

- Trực quan hóa Knowledge Graph

- Thử nghiệm query trên graph lấy ra những nút (thực thể) và cạnh (mối quan hệ) liên quan tới câu queries.

- Đánh giá chất lượng câu trả lời GraphRAG và NaiveRAG trên tập dữ bằng LLM trên các tiêu chứ có sẵn.

**Problems:**

- Multihop RAG (Retrieval-Augmented Generation đa bước) giải quyết một thách thức then chốt trong hệ thống truy xuất thông tin: trong thực tế, phần lớn câu trả lời không nằm gọn trong một đoạn văn đơn lẻ mà cần kết hợp từ nhiều nguồn thông tin phân tán. Điều này yêu cầu hệ thống phải hiểu được mối quan hệ phức tạp giữa các thực thể và cách chúng tương tác với nhau.

**Dữ Liệu:**

- Tập dữ liệu sẽ sử dụng được lấy từ Repo Github sau: [multi-hop RAG task](https://github.com/StonyBrookNLP/musique).

- Thông tin chung về tập dữ liệu:

    + Là bộ dữ liệu multi-hop QA gồm 25K queries mà thông tin để trả lời câu hỏi rải rác từ 2-4 documents được tạo ra bằng cách ghép nhiều câu hỏi single-hop từ các bộ dữ liệu khác.

- Gồm ba file dữ liệu json ở folder musique_dataset là :
    + **./musique_dataset/queries.json**: Gồm tập câu queries

    + **./musique_dataset/corpus.json**: Gồm tập các documents

    + **.musique_dataset/dev_rel_docs.json**: Dữ liệu nối giữa queries và grountruth documents cho queries đó.

# Các bước thực hiện


## Pip install và import những thư viện cần thiết

In [ ]:
!pip install nest_asyncio
!pip install lightrag-hku
!pip install asyncio
!pip install together
!pip install -U FlagEmbedding
!pip install pyvis

KeyboardInterrupt: 

In [ ]:
#mount drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/Practice_Lec7

In [ ]:
!ls

In [ ]:
import os
import json
import sys
sys.path.append("../..")
import nest_asyncio
nest_asyncio.apply()
import os
import asyncio
from lightrag import LightRAG, QueryParam
from lightrag.kg.shared_storage import initialize_pipeline_status
from lightrag.utils import setup_logger

Tập dữ liệu nằm trong folder musique_dataset
Folder gồm ba file là:

- `queries.json`

- `corpus.json`

- `dev_rel_docs.json`


In [ ]:
musique_corpus_file = "./musique_dataset/corpus.json"
musique_queries_file = "./musique_dataset/queries.json"
musique_query_docs_file = "./musique_dataset/dev_rel_docs.json"

## Tiền xử lý dữ liệu

**1. Load dữ liệu từ file json**

In [ ]:
# Đọc dữ liệu hai file:
## Load dữ liệu multi-hop queries từ file multi_hop_rag_file
with open(musique_corpus_file) as f:
    musique_corpus = json.load(f)

## Load corpus từ file multi_hop_corpus_file
with open(musique_queries_file) as f:
    musique_queries = json.load(f)

## Load corpus từ file multi_hop_corpus_file
with open(musique_query_docs_file) as f:
    musique_query_docs = json.load(f)

**2. Lấy 10 queries đầu tiên và lấy những groundtruth documents cho các query đó**

In [ ]:
number_of_queries = 10
sub_musique_queries = {k: v for k, v in
                       list(musique_queries.items())[:number_of_queries]}
total_queries = list(sub_musique_queries.values())

# Tạo sub-corpus bao gồm những văn bản cần để trả lời 10 queries đầu tiên:
all_relevant_doc_ids = set()
for query_id in sub_musique_queries:
    all_relevant_doc_ids.update(musique_query_docs[query_id])

total_corpus = [musique_corpus[doc_id] for doc_id in all_relevant_doc_ids]

# In tổng số lượng văn bản và số lượng queries truy vấn trong corpus
print(f"Tổng số lượng queries: {len(sub_musique_queries)}")
print(f"Tổng số lượng văn bản: {len(total_corpus)}")

Tổng số lượng queries: 10
Tổng số lượng văn bản: 28


## Indexing Graph 🔧

- Trong thư viện lighrag có 4 thành phần phải được xác định:

    + **Mô hình lớn sử dụng**. Mặc định thư viện là gpt-4o từ OpenAI

    + **Mô hình embedding**. Mặc định thư viện là text-embedding-3-small từ OpenAI

    + **Vector database**. Mặc định thư viện là dùng NanoVectorDB

    + **Graph database**. Mặc định thư viện là dùng base trên thư viện networkx

**1️⃣ Custome LLM**

- Gọi LLM từ Together AI thông qua gọi API

- **Together AI 🧠**

    + Là nền tảng cung cấp truy cập đến nhiều mô hình LLM mã nguồn mở

    + Đăng ký tài khoản tại: [TogetherAI](https://www.together.ai/)

    + Mỗi tải khoản mới được free 1$ credits

In [ ]:
# Gọi Api mô hình từ Together.ai

together_api = "tgp_v1_nCs6xSlTOh2iaPvgwuml8pJF6PznAtixRJINqOFTehc"
os.environ["TOGETHER_API_KEY"] = together_api
os.environ["LLM_MODEL_NAME"] = "meta-llama/Llama-3.3-70B-Instruct-Turbo"

In [ ]:
from together import AsyncTogether
from lightrag.base import BaseKVStorage
from lightrag.utils import compute_args_hash

async def together_complete_if_cache(
    model, prompt, system_prompt=None, history_messages=[], **kwargs
) -> str:
    together_client = AsyncTogether(api_key = os.getenv("TOGETHER_API_KEY"))
    hashing_kv: BaseKVStorage = kwargs.pop("hashing_kv", None)
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.extend(history_messages)
    messages.append({"role": "user", "content": prompt})
    if hashing_kv is not None:
        args_hash = compute_args_hash(model, messages)
        if_cache_return = await hashing_kv.get_by_id(args_hash)
        if if_cache_return is not None:
            return if_cache_return["return"]

    response = await together_client.chat.completions.create(
            model=model, messages=messages, **kwargs
    )
    if hashing_kv is not None:
        await hashing_kv.upsert(
            {args_hash: {"return": response.choices[0].message.content,
                          "model": model}}
        )
        await hashing_kv.index_done_callback()
    return response.choices[0].message.content

async def together_llm_complete(
    prompt, system_prompt=None, history_messages=[], **kwargs
) -> str:
    return await together_complete_if_cache(
        os.getenv("LLM_MODEL_NAME"),
        prompt,
        system_prompt=system_prompt,
        history_messages=history_messages,
        **kwargs,
    )

**2️⃣ Custome Embedding**

- Sử dựng mô hình BGE-M3 từ thư viện 🛠️: [FlagEmbedding](https://github.com/FlagOpen/FlagEmbedding).


In [ ]:
from FlagEmbedding import BGEM3FlagModel
from lightrag.utils import wrap_embedding_func_with_attrs
import numpy as np

# Sử dụng embedding model là bge-m3
EMBED_MODEL = BGEM3FlagModel("BAAI/bge-m3",
                       cache_folder="bge-m3",
                       use_fp16=True,
                        devices="cpu")

#BGE-M3 có số chiều là 1024 và set up max tokens là 8192 tokens
embedding_dimension = 1024
max_tokens = 8192

@wrap_embedding_func_with_attrs(
    embedding_dim=embedding_dimension,
    max_token_size=max_tokens,
)
async def bge_m3_embedding(texts: list[str]) -> np.ndarray:
    embeddings = EMBED_MODEL.encode(texts,
                                   return_dense=True)['dense_vecs']
    return embeddings

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

...

onnx/tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

sparse_linear.pt:   0%|          | 0.00/3.52k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

**3️⃣ Xây dựng đồ thị từ văn bản 🏗️**

In [ ]:
total_corpus

list

In [ ]:
setup_logger("lightrag", level="INFO")
WORKING_DIR = "./rag_storage"
if not os.path.exists(WORKING_DIR):
    os.mkdir(WORKING_DIR)
from lightrag import LightRAG
import os
import asyncio
##### TODO: Thực hành #####
# Yêu cầu:
# + Input:
#   - Khởi tạo LightRAG object
#   - Thay thế LLM bằng hàm together_llm_complete() gọi Together AI
#   - Thay thế embedding model bằng BGE-M3 model
#   - Thực hiện quá trình insert các chunks (total_corpus) và xây dựng graph
#   - Quá trình có thể kéo dài 3-5 p
# + Output:
#   - Xây dựng graph thành công và trả về object lightrag
#     để phục vụ truy vấn trên đồ thị sau.
# Tham khảo cách sử dụng tại: https://github.com/HKUDS/LightRAG
##### End TODO #####
async def initialize_rag():
    #######################
    ### START CODE HERE ###
    #######################
    rag = LightRAG(
        working_dir=WORKING_DIR,
        llm_model_func=together_llm_complete,
        embedding_func=bge_m3_embedding
    )
    # IMPORTANT: Both initialization calls are required!
    await rag.initialize_storages()  # Initialize storage backends
    await initialize_pipeline_status()  # Initialize processing pipeline

    return rag

async def insert():
    #######################
    ### START CODE HERE ###
    #######################
    rag = await initialize_rag()

    # Insert dữ liệu (total_corpus phải là list[str])
    rag.insert(total_corpus)

    # Build graph (sẽ mất 3-5 phút tùy số liệu)
    # rag.build_graph()

    return rag



In [ ]:
rag = asyncio.run(insert())


INFO: [_] Loaded graph from ./rag_storage/graph_chunk_entity_relation.graphml with 271 nodes, 281 edges
INFO: Another process is already processing the document queue. Request queued.


In [ ]:
rag


LightRAG(working_dir='./rag_storage', kv_storage='JsonKVStorage', vector_storage='NanoVectorDBStorage', graph_storage='NetworkXStorage', doc_status_storage='JsonDocStatusStorage', workspace='', log_level=None, log_file_path=None, top_k=40, chunk_top_k=20, max_entity_tokens=6000, max_relation_tokens=8000, max_total_tokens=30000, cosine_threshold=0.2, related_chunk_number=5, kg_chunk_pick_method='VECTOR', entity_extract_max_gleaning=1, force_llm_summary_on_merge=8, chunk_token_size=1200, chunk_overlap_token_size=100, tokenizer=<lightrag.utils.TiktokenTokenizer object at 0x785d3124c4a0>, tiktoken_model_name='gpt-4o-mini', chunking_func=<function chunking_by_token_size at 0x785eb0501300>, embedding_func=<function priority_limit_async_func_call.<locals>.final_decro.<locals>.wait_func at 0x785d310ac040>, embedding_batch_num=10, embedding_func_max_async=8, embedding_cache_config={'enabled': False, 'similarity_threshold': 0.95, 'use_llm_check': False}, default_embedding_timeout=30, llm_model_f

- Sau khi xây xong mọi dữ liệu về đồ thị, chunks, vector database được lưu trong **WORKING_DIR**

- **WORKING_DIR** trong bài này được set bằng `./rag_storag`

- Trong folder **WORKING_DIR** gồm:

  + file **graph_chunk_entity_relation.graphml** chứa thông tin về graph.

  + file **kv_store_full_docs.json** chứa thông tin về documents gốc

  + file **kv_store_text_chunks.json** chứa thông tin về các chunks được cắt từ documents

  + file **vdb_chunks.json** lưu embedding của các chunks

  + file **vdb_entities.json** lưu embedding của entities (thực thể).

  + file **vdb_relationships.json** lưu embedding của relationships(mối quan hệ).

## Information of knowledge Graph and Visualizing Graph 🖌️

**1️⃣ Trích xuất một số thông tin về Graphs &#128269;**

   - Số node và edges trong knowledge Graph

   - Lấy ra một node và một cạnh bất kì trong graph và in ra những attributes trong nút/cạnh đó

In [ ]:
import networkx as nx

def print_graph_info(graphml_file: str) -> None:
    ##### TODO: Thực hành #####
    # Yêu cầu:
    # + Input:
    #   - Dùng thư viện networkx để load đồ thị từ file .graphml trong
    #     folder WORIKING_DIR của graphrag
    #   - In ra số lượng nút, số lượng cạnh trong đồ thị
    #   - In ra thông tin của nút và cạnh bất kì trong đồ thị
    # + Output:
    #   - Số lượng nút
    #   - Số lượng cạnh
    #
    #   - Thông tin một nút bất kì gồm: tên nút,
    #     và một số đặc tính của nút như entity type, description
    #     document_id gốc chứa nút thực thể đó
    #
    #   - Thông tin một cạnh bất kì gồm:
    #       + Tên nút đầu của cạnh
    #       + Tên nút cuối cua cạnh
    #       + Trọng số của cạnh
    #       + Description chứa cạnh đó
    #       + document_id gốc chứa nút cạnh đó
    # Tham khảo cách sử dụng tại: https://networkx.org/documentation/stable/tutorial.html

    #######################
    ### START CODE HERE ###
    #######################

    ##### End TODO #####
    graph = nx.read_graphml(graphml_file)
    print("Số lượng nút:", len(graph.nodes()))
    print("Số lượng cạnh:", len(graph.edges()))
    print("Thông tin nút bất kì:")
    for node, attrs in graph.nodes(data=True):
        print(node, attrs)
        break

    print("Thông tin cạnh bất kì:")
    for edge in graph.edges(data=True):
        print(edge)
        break
    return None

graph_path = "./rag_storage/graph_chunk_entity_relation.graphml"
print_graph_info(graph_path)

Số lượng nút: 271
Số lượng cạnh: 281
Thông tin nút bất kì:
Creature {'entity_id': 'Creature', 'entity_type': 'concept', 'description': 'A concept representing a living being, including animals and other organisms.', 'source_id': 'chunk-54d555b5782f3e9e45091d7064f9a964', 'file_path': 'unknown_source', 'created_at': 1762759710, 'truncate': ''}
Thông tin cạnh bất kì:
('Al Saud', 'Najd', {'weight': 1.0, 'description': 'The Al Saud family originated in Najd, central Arabia.', 'keywords': 'location,origin', 'source_id': 'chunk-c6ebe08372383ae0b6cdaa9c889febe0', 'file_path': 'unknown_source', 'created_at': 1762759734, 'truncate': ''})


**2️⃣ Visualize Knowledge Graph đã tạo 📊**

- [**Pyvis**](https://pyvis.readthedocs.io/en/latest/) là một thư viện python để trực quan hóa đồ thị.

- Nó có thể  trực quan hóa đồ thị từ thư viện python quản lý đồ thị như Networkx,...


In [ ]:
import networkx as nx
from pyvis.network import Network
import webbrowser
import ast

# Load file GraphML từ save folder
graphml_file = "./rag_storage/graph_chunk_entity_relation.graphml"
G = nx.read_graphml(graphml_file)

# Khởi tạo Pyvis network
net = Network(
    height="100vh",
    width="100vw",
    bgcolor="black",
    font_color="white",
    notebook=True
)

# Add nodes từ graph của networkx sang Pyvis
for node in G.nodes(data=True):
    node_id, attrs = node
    title = "\n".join([f"{key}: {value}" for key, value in attrs.items()])
    net.add_node(
        node_id,
        label=node_id,
        title=title,
        size=20
    )

for edge in G.edges(data=True):
    source, target, attrs = edge
    title = "\n".join([f"{key}: {value}" for key, value in attrs.items()])
    net.add_edge(
        source,
        target,
        title=title,
        color="#FFFFFF",
        width=2
    )
net.save_graph("graph.html")
# Save and show graph
net.show("graph.html")
webbrowser.open('graph.html')

graph.html


False

## Truy vấn NaiveRAG và truy vấn trên đồ thị

**1️⃣ Lấy ra tập câu sub-queries ở phía trên với groudtruth documents cho mỗi queries**


In [ ]:
query_lists  = sub_musique_queries.values()
query_to_gt_docs = []
for query_id, query_content in sub_musique_queries.items():
    groundtruth_doc_ids = musique_query_docs[query_id]
    grountruth_contents = [musique_corpus[doc_id] for
                            doc_id in groundtruth_doc_ids]
    query_to_gt_docs.append((query_content, grountruth_contents))

**2️⃣ Lấy ra một câu test query bất kì để thực hiện truy vấn**

In [ ]:
test_query, test_gt_documents = query_to_gt_docs[1]
print("Câu queries: ")
print(test_query)
print("Văn bản chứa câu trả lời: ")
for i, doc in enumerate(test_gt_documents):
    print(f"Văn bản: {i}")
    print(doc)

Câu queries: 
Tháng nào thì các cuộc thảo luận Tam bên bắt đầu giữa Anh, Pháp và quốc gia nơi, mặc dù có trụ sở tại quốc gia được gọi là khối thịnh vượng chung của quý tộc, nhưng các đặc vụ Warsaw Pact hàng đầu có nguồn gốc?
Văn bản chứa câu trả lời: 
Văn bản: 0
Szlachta: Giới quý tộc Ba Lan đã tận hưởng nhiều quyền lợi mà không có sẵn cho các giai cấp quý tộc của các quốc gia khác và, điển hình, mỗi vị vua mới đã nhượng bộ cho họ thêm đặc quyền. Những đặc quyền này đã trở thành cơ sở của Tự do Vàng trong Khối thịnh vượng chung Ba Lan - Litva. Mặc dù có một vị vua, Ba Lan được gọi là Khối thịnh vượng chung của quý tộc vì vị vua được bầu bởi tất cả các thành viên quan tâm của quý tộc thừa kế và Ba Lan được coi là tài sản của giai cấp này, chứ không phải của vị vua hay triều đại cai trị. Tình trạng này đã phát triển một phần do sự tuyệt tự của những người thừa kế dòng nam của triều đại hoàng gia cũ (trước hết là Piast, sau đó là Jagiellon) và việc lựa chọn bởi quý tộc của vị vua Ba Lan t

In [ ]:
len(test_gt_documents)

3

**3️⃣ Tìm những chunks liên quan tới query**

In [ ]:
import asyncio
def search_relevant_chunks(query: str,
                           graphrag : LightRAG,
                            top_k = 5) -> list[str]:
    # Lấy ra vector database lưu chunks
    chunk_vdb = graphrag.chunks_vdb

    # Từ chunk_vdb lấy ra top_k chunk_id liên quan đến query nhất
    result = asyncio.run(chunk_vdb.query(query, top_k = top_k))
    result_chunk_id = [sample["__id__"] for sample in result]
    # Load database lưu thông tin về chunk:
    chunk_kv = graphrag.text_chunks

    chunk_objects = asyncio.run(chunk_kv.get_by_ids(result_chunk_id))

    # Trích xuất nội dung của chunk từ chunk object

    chunk_contents = [sample["content"] for sample in chunk_objects]

    return chunk_contents



In [ ]:
print(test_query)

Tháng nào thì các cuộc thảo luận Tam bên bắt đầu giữa Anh, Pháp và quốc gia nơi, mặc dù có trụ sở tại quốc gia được gọi là khối thịnh vượng chung của quý tộc, nhưng các đặc vụ Warsaw Pact hàng đầu có nguồn gốc?


In [ ]:
search_relevant_chunks(test_query, rag, top_k = 10)

INFO: Embedding func: 8 new workers initialized (Timeouts: Func: 30s, Worker: 60s, Health Check: 75s)


['Szlachta: Giới quý tộc Ba Lan đã tận hưởng nhiều quyền lợi mà không có sẵn cho các giai cấp quý tộc của các quốc gia khác và, điển hình, mỗi vị vua mới đã nhượng bộ cho họ thêm đặc quyền. Những đặc quyền này đã trở thành cơ sở của Tự do Vàng trong Khối thịnh vượng chung Ba Lan - Litva. Mặc dù có một vị vua, Ba Lan được gọi là Khối thịnh vượng chung của quý tộc vì vị vua được bầu bởi tất cả các thành viên quan tâm của quý tộc thừa kế và Ba Lan được coi là tài sản của giai cấp này, chứ không phải của vị vua hay triều đại cai trị. Tình trạng này đã phát triển một phần do sự tuyệt tự của những người thừa kế dòng nam của triều đại hoàng gia cũ (trước hết là Piast, sau đó là Jagiellon) và việc lựa chọn bởi quý tộc của vị vua Ba Lan từ trong số những người thừa kế dòng nữ của triều đại.',
 'Hiệp ước Warsaw: Tổ chức của Hiệp ước Warsaw có hai mặt: Ủy ban Tư vấn Chính trị xử lý các vấn đề chính trị, và Bộ Chỉ huy Liên hợp của Lực lượng Vũ trang Hiệp ước kiểm soát các lực lượng đa quốc gia đượ

**4️⃣ Tìm những thực thể  (entities) liên quan tới query**

In [ ]:
import asyncio
def search_relevant_entities(query: str,
                             graphrag : LightRAG,
                             graph_path : str,
                             top_k = 5) -> list[dict[str, str]]:
    # Lấy ra vector database lưu entities
    entities_vdb = graphrag.entities_vdb
    # Từ entities_vdb lấy ra top_k kết quả liên quan đến query nhất
    # từ entities vdb
    result = asyncio.run(entities_vdb.query(query, top_k = top_k))
    print(f"result {result}")
    G = nx.read_graphml(graph_path)
    all_entities_information = []

    ##### TODO: Thực hành #####
    # Yêu cầu:
    # + Input:
    #   - Tìm ra tất cả tên những thực thể liên quan đến query
    #   - Từ tên các thực thể lấy tất cả những thông tin liên quan
    #     đến thực thể đó bao gồm:
    #        + Tên thực thể
    #        + Loại thực thể đó
    #        + Tất cả Miêu tả thực thể
    #        + Bậc của nút thực thể đó
    # + Output:
    #     - Fomart là một list của các dictionaries:
    #          + key là tên đặc điểm
    #          + value là giá trị của đặc điểm đó.
    #     - Ví dụ:
    #         [
    #           {"entity_name" : "...",
    #            "entity_type" : "...",
    #            "description_1" : "...",
    #            "description_2" : "...",
    #            "node_degree" : "..."},
    #            ...
    #         ]
    # Gợi ý: Tham khảo cách sử dụng tại: https://github.com/gusye1234/nano-vectordb
    #                                    https://networkx.org/documentation/stable/tutorial.html

    #######################
    ### START CODE HERE ###
    #######################
    for sample in result:
        # 'sample' là một dictionary.

        # 1. Lấy Tên thực thể (robustly)
        # Ưu tiên lấy từ key 'entity_name'
        entity_name = sample.get('entity_name')

        # Fallback: Nếu không có key 'entity_name', thử parse từ 'content'
        if not entity_name:
            content_string = sample.get('content', '')
            entity_name = content_string.split('\n', 1)[0].strip()

        # Nếu vẫn không có tên, bỏ qua
        if not entity_name:
            continue

        # 2. Lấy thông tin từ Graph (G)
        entity_type = None
        node_degree = 0

        # Kiểm tra xem thực thể có tồn tại trong graph G không
        if entity_name in G:
            # Lấy bậc của nút
            node_degree = G.degree(entity_name)

            # Lấy data của nút (để tìm 'entity_type')
            # node_data là một dict chứa attributes của nút
            node_data = G.nodes[entity_name]

            # Lấy 'entity_type' từ data của nút.
            # Dùng .get() để tránh lỗi nếu key không tồn tại.
            entity_type = node_data.get('entity_type')

        # 3. Lấy Miêu tả (Descriptions)
        # Lấy miêu tả từ trường 'content' của kết quả vector search
        content_string = sample.get('content', '')

        # Tách dòng đầu (thường là tên) khỏi phần còn lại (mô tả)
        content_parts = content_string.split("\n", 1)
        description_text = ""
        if len(content_parts) > 1:
            description_text = content_parts[1] # Lấy toàn bộ phần sau dòng đầu tiên

        # Tách các dòng mô tả
        descriptions = [d.strip() for d in description_text.split("\n") if d.strip()]

        # 4. Tạo dictionary theo format yêu cầu
        info = {
            "entity_name": entity_name,
            "entity_type": entity_type,
            "node_degree": node_degree,
        }

        # 5. Thêm các description_i vào dict
        # (Mô tả này đến từ vectorDB 'content')
        for i, desc in enumerate(descriptions):
            info[f"description_{i+1}"] = desc

        all_entities_information.append(info)
    ##### End TODO #####


    return all_entities_information



In [ ]:
graph_path = "./rag_storage/graph_chunk_entity_relation.graphml"
temp=search_relevant_entities(test_query,
                          rag,
                          graph_path,
                          top_k = 5)

result [{'__id__': 'ent-932bbaca386ee0436ad0159117eabae4', '__created_at__': 1762760365, 'entity_name': 'Anh', 'content': "Anh\nAnh, or the United Kingdom, was involved in the negotiations, expressing concerns over the Soviet Union's proposed language and its potential implications for Finland and the Baltic states.<SEP>Anh là quốc gia nơi diễn ra cuộc chinh phục Norman.", 'source_id': 'chunk-f2e0878e3deb8e67aadac2bcad2c3840<SEP>chunk-af42d1124f930ea854e148b3ba24336e', 'file_path': 'unknown_source', '__metrics__': np.float32(0.49507564), 'id': 'ent-932bbaca386ee0436ad0159117eabae4', 'distance': np.float32(0.49507564), 'created_at': 1762760365}, {'__id__': 'ent-24a5bf049aeef94ab79bad1f73f16b92', '__created_at__': 1762760061, 'entity_name': 'Đức', 'content': 'Đức\nĐức, or Germany, was a central concern in the discussions due to its potential for aggression in Europe.', 'source_id': 'chunk-f2e0878e3deb8e67aadac2bcad2c3840', 'file_path': 'unknown_source', '__metrics__': np.float32(0.460472

In [ ]:
temp

[{'entity_name': 'Anh',
  'entity_type': 'location',
  'node_degree': 2,
  'description_1': "Anh, or the United Kingdom, was involved in the negotiations, expressing concerns over the Soviet Union's proposed language and its potential implications for Finland and the Baltic states.<SEP>Anh là quốc gia nơi diễn ra cuộc chinh phục Norman."},
 {'entity_name': 'Đức',
  'entity_type': 'organization',
  'node_degree': 2,
  'description_1': 'Đức, or Germany, was a central concern in the discussions due to its potential for aggression in Europe.'},
 {'entity_name': 'Warsaw',
  'entity_type': 'location',
  'node_degree': 1,
  'description_1': "Warsaw is the capital city of Poland and the headquarters of the Warsaw Pact's Unified Armed Forces Command."},
 {'entity_name': 'Warsaw Pact',
  'entity_type': 'organization',
  'node_degree': 3,
  'description_1': 'The Warsaw Pact is a political and military alliance with a dual structure, including the Political Consultative Committee and the Unified A

**5️⃣ Tìm những mối quan hệ (relationships) liên quan tới query**

In [ ]:
import asyncio
def search_relevant_relations(query: str,
                              graphrag : LightRAG,
                              graph_path : str,
                              top_k = 5) -> list[dict[str, str]]:
    # Lấy ra vector database lưu relationships
    relationships_vdb = graphrag.relationships_vdb
    # Từ relationships_vdb lấy ra top_k tên thực thể liên quan đến query nhất
    result = asyncio.run(relationships_vdb.query(query, top_k = top_k))
    all_edges_information = []

    ##### TODO: Thực hành #####
    # Yêu cầu:
    # + Input:
    #   - Tương tự bài tập trên tìm tất cả thông tin
    #     cạnh liên quan nhất đến queries
    # + Output:
    #     - Fomart là một list của các dictionaries:
    #          + key là tên đặc điểm
    #          + value là giá trị của đặc điểm đó.
    #     - Ví dụ:
    #         [
    #           {"Node source" : "...",
    #            "Node head" : "...",
    #            "description_1" : "...",
    #            "description_2" : "...",
    #            "Edge weight" : "..."},
    #            ...
    #         ]
    # Tham khảo cách sử dụng tại: https://github.com/gusye1234/nano-vectordb
    #                            https://networkx.org/documentation/stable/tutorial.html

    #######################
    ### START CODE HERE ###
    #######################
    for sample in result:
        # 'sample' là một dictionary từ VectorDB (Relations)
        # Giả định sample chứa 'source' và 'target'

        node_source = sample.get('source')
        node_head = sample.get('target') # 'head' hoặc 'target'

        # Nếu thiếu thông tin cơ bản, bỏ qua
        if not node_source or not node_head:
            continue

        # 1. Lấy Miêu tả (Descriptions) từ VDB
        content_string = sample.get('content', '')
        descriptions = [d.strip() for d in content_string.split("\n") if d.strip()]

        # 2. Lấy thông tin từ Graph (G)
        edge_weight = None

        # Kiểm tra xem cạnh (source, target) có tồn tại trong graph G không
        if G.has_edge(node_source, node_head):
            # Lấy data của cạnh (để tìm 'weight')
            # edge_data là một dict chứa attributes của cạnh
            edge_data = G.edges[node_source, node_head]

            # Lấy 'weight' từ data của cạnh.
            edge_weight = edge_data.get('weight')

        # 3. Tạo dictionary theo format yêu cầu
        info = {
            "Node source": node_source,
            "Node head": node_head,
            "Edge weight": edge_weight,
        }

        # 4. Thêm các description_i vào dict
        # (Mô tả này đến từ VDB 'content')
        for i, desc in enumerate(descriptions):
            info[f"description_{i+1}"] = desc

        all_edges_information.append(info)
    ##### End TODO #####

    return all_edges_information


In [ ]:
graph_path = "./rag_storage/graph_chunk_entity_relation.graphml"
temp=search_relevant_relations(test_query,
                         rag,
                         graph_path, top_k = 5)

In [ ]:
temp

[]

**6️⃣ Trả lời câu query bằng 2 phương pháp là :**

- Naive RAG thông thường

- Phương pháp hybrid(local + global) search trong LightRAG.

In [ ]:
# Naive RAG query
##### TODO: Thực hành #####
    # Yêu cầu:
    # + Input:
    #   - Thực hiện truy vấn và trả ra câu trả lời cho câu test_query.
    #   - Thực hiện truy vấn                                                                                                                                             bằng hai cách là
    #       + naive search (only text chunks)
    #       + mix search   (kết hợp giữa local search và
    #                       global search trên đồ thì)
    # + Output:
    #   Lưu ra câu trả lời của hai phương pháp ra hai biến là:
    #      - naive_answer
    #      - hybrid_answer
    # Gợi ý: Tham khảo cách sử dụng tại: https://github.com/HKUDS/LightRAG

# Naive search

#######################
### START CODE HERE ###
#######################

naive_answer = rag.query(test_query, param=QueryParam(mode="naive"))
print(naive_answer)

# Mix search

#######################
### START CODE HERE ###
#######################
hybrid_answer = rag.query(test_query, param=QueryParam(mode="hybrid"))
print(hybrid_answer)

##### End TODO #####

INFO: Naive query: 20 chunks (chunk_top_k:20 cosine:0.2)
INFO: Final context: 20 chunks
INFO: LLM func: 4 new workers initialized (Timeouts: Func: 180s, Worker: 360s, Health Check: 375s)
INFO:  == LLM cache == saving: naive:query:ff3b9dff1d8fb61d705778b4bff5b4e3


Các cuộc thảo luận Tam bên bắt đầu vào giữa tháng 6. Các cuộc đàm phán này diễn ra giữa Anh và Liên Xô, với Liên Xô là quốc gia có các đặc vụ Warsaw Pact hàng đầu, và mặc dù có trụ sở tại Warsaw, Ba Lan, nhưng các đặc vụ này có nguồn gốc từ Liên Xô. Ba Lan được gọi là "Khối thịnh vượng chung của quý tộc" vì vị vua được bầu bởi tất cả các thành viên quan tâm của quý tộc thừa kế.


INFO:  == LLM cache == saving: hybrid:keywords:6454f820ba13f301d172dbd4fd5a4dc6
INFO: Query nodes: Anh, Pháp, Khối thịnh vượng chung, Warsaw Pact, Quý tộc (top_k:40, cosine:0.2)
INFO: Local query: 40 entites, 47 relations
INFO: Query edges: Cuộc thảo luận Tam bên, Quan hệ quốc tế, Lịch sử chính trị (top_k:40, cosine:0.2)
INFO: Global query: 47 entites, 40 relations
INFO: Raw search results: 71 entities, 71 relations, 0 vector chunks
INFO: After truncation: 71 entities, 71 relations
INFO: Selecting 16 from 16 entity-related chunks by vector similarity
INFO: Find no additional relations-related chunks from 71 relations
INFO: Round-robin merged chunks: 16 -> 16 (deduplicated 0)
INFO: Final context: 71 entities, 71 relations, 16 chunks
INFO: Final chunks S+F/O: E8/1 E11/2 E10/3 E2/4 E5/5 E8/6 E1/7 E12/8 E2/9 E2/10 E6/11 E1/12 E2/13 E6/14 E3/15 E2/16
INFO:  == LLM cache == saving: hybrid:query:07d9de25f258b8ea6998dedcb7f17378


Các cuộc thảo luận Tam bên bắt đầu vào giữa tháng 6. Các cuộc thảo luận này liên quan đến Anh và Liên Xô, với Liên Xô đề xuất xem xét rằng một bước ngoặt chính trị hướng tới Đức của các quốc gia Baltic sẽ cấu thành một "cuộc tấn công gián tiếp" vào Liên Xô. Quốc gia được đề cập, nơi có trụ sở của Warsaw Pact nhưng có các đặc vụ hàng đầu của Warsaw Pact có nguồn gốc từ một quốc gia khác, ám chỉ đến Liên Xô vì các vị trí then chốt trong Warsaw Pact được nắm giữ bởi các quan chức Liên Xô.


## Đánh giá câu trả lời của Naive RAG và GraphRAG bằng LLM 👨‍🔬

- Trong phần này, ta sẽ dùng LLM để đánh giá hai câu trả lời của NaiveRAG và LightRAG Hybrid dựa trên các tiêu chí sau:

    + **Tính toàn diện (Comprehensiveness) 📖**: Đánh giá xem câu trả lời có thể trả lời câu hỏi 1 cách toàn diện hay không

    + **Tính đa dạng (Diversity) 🌐** : Đánh giá xem câu trả lời có đa dạng và đưa nhiều thông tin đa chiều về câu hỏi không

    + **Empowerment** 🙋: Câu trả lời giúp người đọc hiểu và đưa ra đánh giá sáng suốt về chủ đề như thế nào ?

In [ ]:
from prompt.evaluation_prompt import EVALUATION_PROMPT
# In ra prompt để đánh giá giữa 2 câu trả lời
print(EVALUATION_PROMPT)


---Role---
You are an expert tasked with evaluating two answers to the same question based on three criteria: **Comprehensiveness**, **Diversity**, and **Empowerment**.
---Goal---
You will evaluate two answers to the same question based on three criteria: **Comprehensiveness**, **Diversity**, and **Empowerment**.

- **Comprehensiveness**: How much detail does the answer provide to cover all aspects and details of the question?
- **Diversity**: How varied and rich is the answer in providing different perspectives and insights on the question?
- **Empowerment**: How well does the answer help the reader understand and make informed judgments about the topic?

For each criterion, choose the better answer (either Answer 1 or Answer 2) and explain why. Then, select an overall winner based on these three categories.

Here is the question:
{query}

Here is the documents containing groundtruth:
{documents}

Here are the two answers:

**Answer 1:**
{answer1}

**Answer 2:**
{answer2}

Evaluate b

In [ ]:
from together import Together
def evaluate_answer(answer_1: str,
                     answer_2: str,
                     query: str,
                     gt_docs: list[str]) -> str:
    client = AsyncTogether(api_key=together_api)
    MODEL_NAME = "meta-llama/Meta-Llama-3.1-70B-Instruct-Turbo" # Hoặc model bạn muốn dùng
    ##### TODO: Thực hành #####
    # Yêu cầu:
    # + Input:
    #   - Từ câu trả lời 1, câu trả lời 2, query
    #     và documents chứa đoạn thông tin chính xác
    #
    #   - Đánh giá câu trả lời nào tốt hơn dựa trên 3 tiêu chí

    #   - Sử dụng EVALUATION_PROMPT ở phía trên kèm theo 2 câu trả lời
    #     sinh ra bởi naive và mix search, đánh giá 2 câu trả lời đó

    #   - Sử dụng mô hình LLM từ Together AI để đánh giá

    #   - Gợi ý: Nên dùng mô hình LLM to ví dụ meta-llama/Llama-3.3-70B-Instruct-Turbo
    # + Output:
    #    - Text đánh giá nhận xét về 2 câu trả lời do LLM sinh ra
    # Gợi ý: Tham khảo cách sử dụng tại: https://docs.together.ai/docs/quickstart

    #######################
    ### START CODE HERE ###
    #######################
    # input answer_1, answer_2, query, gt_docs to EVALUATION_PROMPT
    prompt = EVALUATION_PROMPT.format(
        answer1=answer_1,
        answer2=answer_2,
        query=query,
        documents="\n".join(gt_docs)
    )
    return together_llm_complete(prompt, temperature = 0.0)





In [ ]:
result = asyncio.run(evaluate_answer(naive_answer, hybrid_answer, test_query, test_gt_documents))
print(result)

{
    "Comprehensiveness": {
        "Winner": "Answer 2",
        "Explanation": "Answer 2 cung cấp nhiều chi tiết hơn về các cuộc thảo luận Tam bên, bao gồm cả đề xuất của Liên Xô về việc xem xét một bước ngoặt chính trị hướng tới Đức của các quốc gia Baltic sẽ cấu thành một 'cuộc tấn công gián tiếp' vào Liên Xô. Điều này cho thấy Answer 2 có sự bao quát tốt hơn về các khía cạnh của câu hỏi. Ngoài ra, Answer 2 cũng giải thích rõ hơn về mối quan hệ giữa Liên Xô và Warsaw Pact, giúp người đọc hiểu rõ hơn về vấn đề."
    },
    "Diversity": {
        "Winner": "Answer 2",
        "Explanation": "Answer 2 cung cấp nhiều góc nhìn và thông tin đa dạng hơn về các cuộc thảo luận Tam bên và mối quan hệ giữa các quốc gia. Answer 2 đề cập đến việc Liên Xô đề xuất xem xét một bước ngoặt chính trị hướng tới Đức của các quốc gia Baltic, điều này cho thấy sự đa dạng trong cách tiếp cận và giải quyết vấn đề. Ngoài ra, Answer 2 cũng giải thích rõ hơn về vai trò của Liên Xô trong Warsaw Pact, giúp ngư

In [ ]:
pip install tqdm nest_asyncio

In [ ]:
import asyncio
import json
import re
from tqdm.asyncio import tqdm_asyncio

# --- 1. HÀM HỖ TRỢ PARSE JSON TỪ LLM ---
def clean_json_string(json_str):
    """Làm sạch chuỗi JSON trả về từ LLM nếu nó chứa markdown"""
    if "```json" in json_str:
        json_str = json_str.split("```json")[1].split("```")[0]
    elif "```" in json_str:
        json_str = json_str.split("```")[1].split("```")[0]
    return json_str.strip()

# --- 2. HÀM XỬ LÝ 1 CẶP CÂU HỎI (WRAPPER) ---
async def process_single_evaluation(idx, query, gt_docs, rag_func_1, rag_func_2, eval_func, semaphore):
    """
    Chạy sinh câu trả lời từ 2 model và gọi hàm đánh giá cho 1 query.
    """
    async with semaphore: # Giới hạn số luồng chạy cùng lúc
        try:
            # Bước 1: Sinh câu trả lời từ 2 hệ thống RAG (nếu chưa có sẵn)
            # Giả định rag_func_1 và rag_func_2 là hàm async.
            # Nếu chúng là hàm thường (sync), dùng: ans1 = rag_func_1(query)
            ans1 = await rag_func_1(query)
            ans2 = await rag_func_2(query)

            # Bước 2: Gọi hàm đánh giá (đã có từ trước)
            eval_result_str = await eval_func(ans1, ans2, query, gt_docs)

            # Bước 3: Parse kết quả str thành dict
            try:
                cleaned_str = clean_json_string(eval_result_str)
                eval_json = json.loads(cleaned_str)
            except json.JSONDecodeError:
                # Fallback nếu LLM trả về lỗi format
                eval_json = {"Error": "Invalid JSON format from LLM", "Raw": eval_result_str}

            return {
                "id": idx,
                "query": query,
                "answer1": ans1, # Lưu lại để đối chiếu
                "answer2": ans2,
                "evaluation": eval_json
            }
        except Exception as e:
            print(f"❌ Lỗi tại query {idx}: {e}")
            return None

# --- 3. HÀM CHẠY BATCH CHÍNH ---
async def run_batch_eval(query_gt_pairs, rag1_func, rag2_func, evaluator_func, max_concurrency=5):
    """
    query_gt_pairs: list các tuple (query, gt_docs)
    max_concurrency: số lượng đánh giá chạy song song (tùy rate limit key của bạn)
    """
    sem = asyncio.Semaphore(max_concurrency)
    tasks = []

    print(f"🚀 Bắt đầu batch eval cho {len(query_gt_pairs)} truy vấn...")

    for i, (query, gt_docs) in enumerate(query_gt_pairs):
        task = process_single_evaluation(
            i, query, gt_docs, rag1_func, rag2_func, evaluator_func, sem
        )
        tasks.append(task)

    # Sử dụng tqdm_asyncio để hiện thanh tiến trình
    results = await tqdm_asyncio.gather(*tasks, desc="Đang đánh giá")

    # Lọc bỏ các kết quả lỗi (None)
    valid_results = [r for r in results if r is not None]
    print("✅ Hoàn tất batch evaluation.")
    return valid_results

In [ ]:
# --- ĐỊNH NGHĨA CÁC HÀM RAG CỦA BẠN (VÍ DỤ) ---
# Giả sử bạn đã có object `naive_rag` và `light_rag`

async def get_naive_answer(query):
    # Thay bằng code gọi Naive RAG thật của bạn
    # ví dụ: return await naive_rag.query(query)
    return rag.query(test_query, param=QueryParam(mode="naive"))

async def get_hybrid_answer(query):
    # Thay bằng code gọi LightRAG thật của bạn
    # ví dụ: return await light_rag.query(query, param=...)
    return rag.query(test_query, param=QueryParam(mode="hybrid"))

# --- CHẠY BATCH ---
# 1. Chuẩn bị dữ liệu input (list các cặp query và ground truth)
# Giả sử query_to_gt_docs của bạn là list các tuple: [("query1", [doc1, doc2]), ("query2", [doc3])]
test_data = query_to_gt_docs

# 2. Gọi chạy (nếu trong Colab/Jupyter đã có sẵn event loop)
# Lưu ý: evaluate_answer là hàm đánh giá bạn đã viết ở các bước trước
batch_results = await run_batch_eval(
    query_gt_pairs=test_data,
    rag1_func=get_naive_answer,    # Model đại diện cho "Answer 1"
    rag2_func=get_hybrid_answer, # Model đại diện cho "Answer 2"
    evaluator_func=evaluate_answer, # Hàm LLM Judge của bạn
    max_concurrency=3 # Điều chỉnh số này tùy vào độ mạnh key của bạn
)

# 3. Lưu kết quả ra file
with open("final_evaluation_results.json", "w", encoding="utf-8") as f:
    json.dump(batch_results, f, ensure_ascii=False, indent=2)
print("Đã lưu kết quả vào final_evaluation_results.json")

🚀 Bắt đầu batch eval cho 10 truy vấn...


Đang đánh giá:   0%|          | 0/10 [00:00<?, ?it/s]INFO: Naive query: 20 chunks (chunk_top_k:20 cosine:0.2)
INFO: Final context: 20 chunks
INFO:  == LLM cache == Query cache hit, using cached response as query result
INFO: Naive query: 20 chunks (chunk_top_k:20 cosine:0.2)
INFO: Final context: 20 chunks
INFO:  == LLM cache == Query cache hit, using cached response as query result
INFO: Naive query: 20 chunks (chunk_top_k:20 cosine:0.2)
INFO: Final context: 20 chunks
INFO:  == LLM cache == Query cache hit, using cached response as query result
INFO: Query nodes: Anh, Pháp, Khối thịnh vượng chung, Warsaw Pact, Quý tộc (top_k:40, cosine:0.2)
INFO: Local query: 40 entites, 47 relations
INFO: Query edges: Cuộc thảo luận Tam bên, Quan hệ quốc tế, Lịch sử chính trị (top_k:40, cosine:0.2)
INFO: Global query: 47 entites, 40 relations
INFO: Raw search results: 71 entities, 71 relations, 0 vector chunks
INFO: After truncation: 71 entities, 71 relations
INFO: Selecting 16 from 16 entity-related 

✅ Hoàn tất batch evaluation.
Đã lưu kết quả vào final_evaluation_results.json


In [ ]:
def analyze_winner(results):
    counts = {"Answer 1 (Naive)": 0, "Answer 2 (LightRAG)": 0, "Tie/Unknown": 0}

    for res in results:
        try:
            winner = res['evaluation']['Overall Winner']['Winner'].lower()
            if "answer 1" in winner:
                counts["Answer 1 (Naive)"] += 1
            elif "answer 2" in winner:
                counts["Answer 2 (LightRAG)"] += 1
            else:
                counts["Tie/Unknown"] += 1
        except (KeyError, AttributeError):
            counts["Tie/Unknown"] += 1

    print("\n--- 📊 KẾT QUẢ TỔNG HỢP ---")
    for key, value in counts.items():
        print(f"{key}: {value}")

    total = sum(counts.values())
    if total > 0:
        print(f"Win rate Answer 2: {(counts['Answer 2 (LightRAG)']/total)*100:.1f}%")

# Gọi hàm phân tích
analyze_winner(batch_results)


--- 📊 KẾT QUẢ TỔNG HỢP ---
Answer 1 (Naive): 0
Answer 2 (LightRAG): 2
Tie/Unknown: 8
Win rate Answer 2: 20.0%
